In [1]:
import pandas as pd

data = pd.read_json("data/data.jsonl", lines=True)
data.head(10)

,arabic,arabizi,label
0,وي انا تعجبني برشا,Oui ene t3jbni barcha,pos
1,تبارك الله عليه,tbarek allah 3lih,pos
2,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,imra2a ani9a w salisa fi 5itabiha w dhakiya w ...,pos
3,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,ja3far y7eb bladou w ya3mel fil 5ir y3awen fi ...,pos
4,محلاها,Mahleha,pos
5,عماد دغيج برافو والله راجل موش كيما بعض السياس...,3imed d8ij bravo wlh rajel mouch kima ba3th es...,pos
6,ليلوشاااااااا محلاك تهبل تقتتتتتل تقتل <3,Laylouchaaaaaaaa mahlek t-habel to9999tel to9t...,pos
7,مرا تونسية بأتم معنى الكلمة,Mra tounsia b2tam ma3na lklima,pos
8,معلم,Ma3alme,pos
9,تعجبني برشا,te3jebni barcha,pos


In [2]:
def build_corpus(data,column):
    corpus =""
    for row in data.iterrows():
        corpus += row[1][column] + " "
    return corpus

In [3]:
#Text Analysing
#check arabic characters
arabic_corpus = build_corpus(data,"arabic")
arabic_chars = "".join(sorted(set([char for char in arabic_corpus])))
arabic_chars
#some characters are repeated , so they have different unicode values, we need to keep one unicode for all of them
# -> the fix is NFKC normalization
#We have Emojis (are not needed in the seq2seq model)  -> the fix is to remove them




' !"#%&\'()*+,-./0123456789:;<=>?@BDOPXY\\^_acdeiklmnopsvwxy|~¡°µ·¿þ،؛؟ءآأؤإئابةتثجحخدذرزسشصضطظعغـفقكلمنهوىيًٌٍَُِّْٓ٠١٢٪ٰٱپڤڥڨکگھیەۚ\u200d‘’•…‼√⊙╭╮╯╰▬○●☆☝☺♡♥♪⛽✊✋✌❌❤️ﺃﺇﺍﺎﺏﺐﺑﺒﺓﺔﺖﺗﺘﺝﺞﺟﺢﺣﺤﺨﺩﺪﺬﺭﺮﺯﺰﺳﺴﺵﺷﺸﺼﻀﻄﻉﻋﻌﻏﻒﻓﻔﻖﻗﻚﻛﻜﻞﻟﻠﻡﻢﻣﻤﻥﻧﻨﻩﻪﻫﻬﻭﻮﻲﻳﻴﻵﻷﻹﻻ�🇳🇹🌸🌹🌺🌻🍉🎈🎉🎋🎤🎩🏻🏼🏿🐋🐝🐟🐡🐬🐳🐺👄👈👊👋👌👍👎👏👑👙👣👧👱👴👹👺💇💋💐💓💔💕💖💗💘💙💚💛💜💝💞💟💣💩💪💸🔥🔪🔫😀😁😂😃😄😅😆😇😈😉😊😋😍😎😏😑😒😓😔😕😖😘😙😚😜😝😞😠😡😢😣😤😥😦😧😨😩😪😬😭😮😯😱😲😳😴😵😸😹😻🙇🙈🙉🙊🙌🙏🚯🤓🤗'

In [4]:
#Preprocessing the arabic text
#normalization function (written by claude)
import re, unicodedata

TASHKEEL = re.compile(r"[\u064B-\u0652\u0670]")

# Allowed: Arabic letters ء–ي + Tunisian ڤ ڨ, digits 0-9, whitespace, punctuation
NOT_ALLOWED = re.compile(
    r"[^\u0621-\u063A\u0641-\u064A\u06A4\u06A8"   # Arabic letters (ـ excluded) + ڤ ڨ
    r"0-9\s"                                       # digits + spaces
    r"!\"%'()*,\-./:;?\[\]_…،؛؟]"                 # punctuation (Latin + Arabic)
)
PUNCT_TO_DOT = re.compile(r"[!?;,،؛؟]+")

def normalize_ar(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)                          # presentation forms → standard
    text = text.replace("ی", "ي").replace("ک", "ك")                     # Persian → Arabic
    text = TASHKEEL.sub("", text)                                       # remove diacritics
    text = text.replace("ـ", "")                                        # remove tatweel
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")) 
    text = PUNCT_TO_DOT.sub(".", text)  # Arabic-Indic → 0-9
    text = NOT_ALLOWED.sub(" ", text)                                   # drop everything else
    return re.sub(r"\s+", " ", text).strip()                            # collapse spaces

In [5]:
normilized_corpus = normalize_ar(arabic_corpus)
norm_arabic_chars = "".join(sorted(set([char for char in normilized_corpus])))
print(norm_arabic_chars)
#good results i will apply it to the whole dataset and save it in a new column
data["normilized_arabic"] = data["arabic"].map(normalize_ar)

 "%'()*-./0123456789:_ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىيڤڨ


In [6]:
data.head()
#looks exactly the same

,arabic,arabizi,label,normilized_arabic
0,وي انا تعجبني برشا,Oui ene t3jbni barcha,pos,وي انا تعجبني برشا
1,تبارك الله عليه,tbarek allah 3lih,pos,تبارك الله عليه
2,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,imra2a ani9a w salisa fi 5itabiha w dhakiya w ...,pos,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة
3,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,ja3far y7eb bladou w ya3mel fil 5ir y3awen fi ...,pos,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...
4,محلاها,Mahleha,pos,محلاها


In [7]:
#Text Analysing
#check arabizi characters

arabizi_corpus = build_corpus(data,"arabizi")
arabizi_chars = "".join(sorted(set([char for char in arabizi_corpus])))
arabizi_chars
#we need to lowercase it , remove emojis ,and those char: ⊙○●🏿 , accesnts éèà doesn't have any meaning


' !"#%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ\\^_`abcdefghijklmnopqrstuvwxyz|~¡°µ·¿ÈÉàáâçèéêìíîïôûýþčėęĚł‘’•…‼√⊙╭╮╯╰▬○●☆☝☺♡♥♪⛽✊✋✌❌❤️�🇳🇹🌸🌹🌺🌻🍉🎈🎉🎋🎤🎩🏻🏼🏿🐋🐝🐟🐡🐬🐳🐺👄👈👊👋👌👍👎👏👑👙👣👧👱👴👹👺💇💋💐💓💔💕💖💗💘💙💚💛💜💝💞💟💣💩💪💸🔥🔪🔫😀😁😂😃😄😅😆😇😈😉😊😋😍😎😏😑😒😓😔😕😖😘😙😚😜😝😞😠😡😢😣😤😥😦😧😨😩😪😬😭😮😯😱😲😳😴😵😸😹😻🙇🙈🙉🙊🙌🙏🚯🤓🤗'

In [8]:
#preprocessing the arabizi text
#function written by claude


# Allowed: a-z, digits (3/5/7/9/8 are letters in Arabizi!), whitespace, punctuation
NOT_ALLOWED = re.compile(r"[^a-z0-9\s!\"%'()*,\-./:;?\[\]_]")

def normalize_arabizi(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).replace("…", "...")
    text = unicodedata.normalize("NFKD", text)                          # é → e + accent mark
    text = "".join(c for c in text if not unicodedata.combining(c))     # drop accent marks
    text = text.lower()     
    text = PUNCT_TO_DOT.sub(".", text)                                            # lowercase
    text = NOT_ALLOWED.sub(" ", text)                                   # drop emojis, ⊙○●🏿, <>=^ …
    return re.sub(r"\s+", " ", text).strip()                            # collapse spaces

In [9]:
normilized_arabizi_corpus = normalize_arabizi(arabizi_corpus)
print(normilized_arabizi_corpus[:100])
print("".join(sorted(set([char for char in normilized_arabizi_corpus]))))
#looks good

oui ene t3jbni barcha tbarek allah 3lih imra2a ani9a w salisa fi 5itabiha w dhakiya w motawadhi3a ja
 "%'()*-./0123456789:_abcdefghijklmnopqrstuvwxyz


In [10]:
data['normilized_arabizi'] = data['arabizi'].map(normalize_arabizi)
data.head()

,arabic,arabizi,label,normilized_arabic,normilized_arabizi
0,وي انا تعجبني برشا,Oui ene t3jbni barcha,pos,وي انا تعجبني برشا,oui ene t3jbni barcha
1,تبارك الله عليه,tbarek allah 3lih,pos,تبارك الله عليه,tbarek allah 3lih
2,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,imra2a ani9a w salisa fi 5itabiha w dhakiya w ...,pos,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,imra2a ani9a w salisa fi 5itabiha w dhakiya w ...
3,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,ja3far y7eb bladou w ya3mel fil 5ir y3awen fi ...,pos,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,ja3far y7eb bladou w ya3mel fil 5ir y3awen fi ...
4,محلاها,Mahleha,pos,محلاها,mahleha


In [11]:
preprocessed_data = data[['normilized_arabic','normilized_arabizi',"label"]].rename(
    columns={"normilized_arabic": "src", "normilized_arabizi": "tgt"}
)


In [12]:
preprocessed_data.iloc[6]
#some letters are repeated more than 2 which doesn't add any meaning in context of seq2seq model, so we will remove them and keep only 2 of them

src               ليلوشاااااااا محلاك تهبل تقتتتتتل تقتل 3
tgt      laylouchaaaaaaaa mahlek t-habel to9999tel to9t...
label                                                  pos
Name: 6, dtype: str

In [13]:
arabizi_ELONG = re.compile(r"([^\d\s]|[235789])\1{2,}")
arabic_ELONG = re.compile(r"([^\d\s]|[235789])\1+")

def arabizi_reduce_elongation(text: str) -> str:
    return arabizi_ELONG.sub(r"\1\1", text)

def arabic_reduce_elongation(text: str) -> str:
    return arabic_ELONG.sub(r"\1", text)

preprocessed_data["src"] = preprocessed_data["src"].map(arabic_reduce_elongation)
preprocessed_data["tgt"] = preprocessed_data["tgt"].map(arabizi_reduce_elongation)

preprocessed_data.iloc[6]


src                   ليلوشا محلاك تهبل تقتل تقتل 3
tgt      laylouchaa mahlek t-habel to99tel to9tel 3
label                                           pos
Name: 6, dtype: str

In [14]:
preprocessed_data.to_json("data/preprocessed_data.jsonl", orient="records", lines=True,force_ascii=False)
preprocessed_data.head()

,src,tgt,label
0,وي انا تعجبني برشا,oui ene t3jbni barcha,pos
1,تبارك اله عليه,tbarek allah 3lih,pos
2,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,imra2a ani9a w salisa fi 5itabiha w dhakiya w ...,pos
3,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,ja3far y7eb bladou w ya3mel fil 5ir y3awen fi ...,pos
4,محلاها,mahleha,pos


In [15]:
len(preprocessed_data)

17054

In [16]:
#preparing the data for seq2seq model
#first we need to divide our data into training and testing sets (ennd ensure that no dupplicate rows are in both sets)

preprocessed_data = preprocessed_data[preprocessed_data.src != preprocessed_data.tgt]
preprocessed_data = preprocessed_data.drop_duplicates(subset=["src", "tgt"])

# drop very long lines (some are 1000 chars) and badly mismatched pairs
preprocessed_data = preprocessed_data[preprocessed_data.src.str.len().between(1, 200) & preprocessed_data.tgt.str.len().between(1, 200)]
ratio = preprocessed_data.tgt.str.len() / preprocessed_data.src.str.len()
preprocessed_data = preprocessed_data[ratio.between(0.4, 1.6)]


train_data = preprocessed_data.sample(frac=0.8, random_state=42)
test_data = preprocessed_data.drop(train_data.index)


In [17]:
print(f"Length of preprocessed data: {len(preprocessed_data)}")
print(f"Length of training data: {len(train_data)}")
print(f"Length of testing data: {len(test_data)}")

Length of preprocessed data: 14404
Length of training data: 11523
Length of testing data: 2881


In [18]:
train_data.to_json("data/train_data.jsonl", orient="records", lines=True,force_ascii=False)
test_data.to_json("data/test_data.jsonl", orient="records", lines=True,force_ascii=False)


In [19]:
def encode(seq,stoi,special_tokens=True):
    if special_tokens:
        seq = ["<sos>"] + list(seq) + ["<eos>"]
    return [stoi.get(char, stoi["<unk>"]) for char in seq]

In [20]:
emb = []
for char in 'bravo':
    emb.append(arabizi_stoi.get(char, arabizi_stoi["<unk>"]))
print(emb)

NameError: name 'arabizi_stoi' is not defined

In [ ]:
encode('bravo bravo', arabizi_stoi, special_tokens=False)

[27, 43, 26, 47, 40, 4, 27, 43, 26, 47, 40]

In [ ]:
encode('برافو', arabic_stoi, special_tokens=True)

[1, 33, 42, 32, 52, 59, 2]

In [ ]:
train_data['arabic_length'] = train_data['src'].apply(len)
train_data['arabizi_length'] = train_data['tgt'].apply(len)

train_data.describe()



,arabic_length,arabizi_length
count,11523.000000,11523.000000
mean,31.796147,39.405016
std,27.744858,34.634204
min,1.000000,1.000000
25%,13.000000,16.000000
50%,22.000000,28.000000
75%,41.000000,51.000000
max,171.000000,200.000000


In [ ]:
#the max length of arabic and arabizi sequences is 200, adding <sos> and <eos> tokens will make the max length 202 but the mean is ~ 32
#so it is going to make our model very slow if we pad all sequences to length 202, so we will pad the sequences to length ~ 170


MAX_SRC = 170
MAX_TGT = 172
